# Exploratory Data Analysis — ALPR Dataset Pipeline

This notebook walks through Part 1 (Inspection) and Part 2 (EDA) of the pipeline.
It scans the raw datasets, computes per-image statistics, detects duplicates,
and generates publication-quality figures saved to `reports/figures/`.

**Outputs:**
- `reports/figures/<dataset_name>/*.png/.svg` — 22 figure types per dataset
- `reports/figures/dataset_size_comparison.png/.svg` — cross-dataset bar chart
- `reports/figures/<dataset_name>_eda_report.md` — per-dataset summary report
- `reports/eda/dataset_summary.md/.csv` — inspection summary

**Requirements:** `pip install -r requirements.txt`

In [1]:
from __future__ import annotations
import sys
from pathlib import Path

# --- path setup ---
def _find_project_root(marker="pyproject.toml"):
    path = Path.cwd().resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    return path

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120

print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.version}")

Project root: C:\Users\Admin\Documents\GitHub\AI-Tools-Project
Python: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


---
## 1. Load Configuration

In [2]:
from alpr_dataset.config import PipelineConfig
from alpr_dataset.logging_setup import setup_logging
from alpr_dataset.io_utils import list_images

config = PipelineConfig.load(
    PROJECT_ROOT / "configs" / "pipeline_config.yaml",
    PROJECT_ROOT / "configs" / "datasets.yaml",
)
logger = setup_logging(config.logs_dir, name="alpr_dataset")

print(f"Datasets configured: {[s.name for s in config.datasets]}")
print(f"Reports dir: {config.reports_dir}")
print(f"Blur threshold: {config.blur_threshold}")
print(f"Duplicate hash threshold: {config.duplicate_hash_threshold}")

Datasets configured: ['dataset_A', 'dataset_B']
Reports dir: C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports
Blur threshold: 100.0
Duplicate hash threshold: 5


---
## 2. Part 1 — Dataset Inspection

Scan folder structure, count files, detect formats, check for orphans.

In [3]:
from alpr_dataset.inspection.scanner import scan_dataset
from alpr_dataset.inspection.report import generate_dataset_summary

dataset_specs = [(s.name, s.root) for s in config.datasets]
df_summary = generate_dataset_summary(
    dataset_specs,
    output_dir=config.reports_dir / "eda",
    hamming_threshold=config.duplicate_hash_threshold,
)
df_summary

[07/08/26 10:31:00] INFO     Scanned 'dataset_A': 464 images, 462 annotations, 0 unsupported files

[dataset_A] image stats:   0%|          | 0/464 [00:00<?, ?it/s]

[dataset_A] image stats:   1%|          | 3/464 [00:00<01:01,  7.47it/s]

[dataset_A] image stats:   1%|          | 4/464 [00:00<01:38,  4.67it/s]

[dataset_A] image stats:   1%|          | 5/464 [00:01<02:02,  3.75it/s]

...

[dataset_B] image stats: 100%|█████████▉| 2083/2087 [00:47<00:00, 61.68it/s]

[07/08/26 10:33:50] WARNING  Could not compute phash for                                                           
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\raw\dataset_B\Vehicles\2018.jpg:
                             image file is truncated (5 bytes not processed)

[07/08/26 10:33:53] INFO     Wrote dataset summary CSV ->                                                          
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\eda\dataset_summary.csv

                    INFO     Wrote dataset summary Markdown ->                                                     
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\eda\dataset_summary.md

,dataset_name,root,n_images,n_annotations,n_images_missing_annotations,n_orphan_annotations,n_unsupported_files,n_corrupted_images,n_duplicate_filenames_groups,n_exact_duplicate_images_groups,...,annotation_formats,color_space_counts,avg_width,avg_height,min_resolution_wh,max_resolution_wh,avg_aspect_ratio,avg_file_size_kb,min_file_size_kb,max_file_size_kb
0,dataset_A,C:\Users\Admin\Documents\GitHub\AI-Tools-Proje...,464,462,2,0,0,0,0,0,...,{'.xml': 462},{'RGB': 464},1848.97,2420.55,256x226,3456x4608,0.879,2791.30,18.77,8867.87
1,dataset_B,C:\Users\Admin\Documents\GitHub\AI-Tools-Proje...,2087,2088,0,1,0,1,0,4,...,{'.txt': 2088},{'RGB': 2086},883.43,904.12,240x249,4032x3024,1.032,96.46,9.71,1149.10


In [4]:
# Per-dataset folder tree
for spec in config.datasets:
    scan = scan_dataset(spec.name, spec.root)
    print(f"\n{'='*60}\n{spec.name}\n{'='*60}")
    print(f"Images: {scan.n_images}  |  Annotations: {scan.n_annotations}")
    print(f"Image formats: {dict(scan.image_format_counts)}")
    print(f"Annotation formats: {dict(scan.annotation_format_counts)}")
    print(f"Missing annotations: {len(scan.images_missing_annotations)}")
    print(f"Orphan annotations: {len(scan.orphan_annotations)}")

                    INFO     Scanned 'dataset_A': 464 images, 462 annotations, 0 unsupported files


dataset_A
Images: 464  |  Annotations: 462
Image formats: {'.jpg': 464}
Annotation formats: {'.xml': 462}
Missing annotations: 2
Orphan annotations: 0


                    INFO     Scanned 'dataset_B': 2087 images, 2088 annotations, 0 unsupported files


dataset_B
Images: 2087  |  Annotations: 2088
Image formats: {'.jpg': 2085, '.jpeg': 2}
Annotation formats: {'.txt': 2088}
Missing annotations: 0
Orphan annotations: 1


---
## 3. Load Annotations

Parses each dataset's annotations into the unified `ImageAnnotation` schema.

In [5]:
from tqdm import tqdm
from alpr_dataset.annotations.loader import load_dataset_annotations

all_annotations = {}
for spec in config.datasets:
    annots = load_dataset_annotations(spec)
    all_annotations[spec.name] = annots
    n_boxes = sum(a.n_boxes for a in annots)
    print(f"{spec.name}: {len(annots)} images annotated, {n_boxes} bounding boxes total")

dataset_A: 231 images annotated, 272 bounding boxes total


[07/08/26 10:34:28] WARNING  Skipping unreadable image for YOLO annotation:                                        
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\raw\dataset_B\Vehicles\2018.jpg

dataset_B: 2086 images annotated, 2142 bounding boxes total


In [6]:
# Quick peek at annotation structure
for name, annots in all_annotations.items():
    if annots:
        a = annots[0]
        print(f"\n{name} — first annotation:")
        print(f"  Image: {a.image_path.name} ({a.image_width}x{a.image_height})")
        print(f"  Boxes: {a.n_boxes}")
        if a.boxes:
            b = a.boxes[0]
            print(f"  First box: class={b.class_name} [{b.x_min:.0f},{b.y_min:.0f}]->[{b.x_max:.0f},{b.y_max:.0f}]")
            print(f"  Box width={b.width:.0f}, height={b.height:.0f}, area={b.area:.0f}")


dataset_A — first annotation:
  Image: IMG20221107210304.jpg (3456x4608)
  Boxes: 1
  First box: class=1 [1478,1465]->[1801,1646]
  Box width=323, height=181, area=58463

dataset_B — first annotation:
  Image: 0001.jpg (720x734)
  Boxes: 1
  First box: class=license_plate [333,372]->[445,430]
  Box width=112, height=58, area=6496


---
## 4. Compute Image Statistics

Per-image: resolution, brightness, contrast, blur, sharpness, entropy.

In [7]:
from alpr_dataset.inspection.image_stats import batch_compute_stats
from alpr_dataset.inspection.hashing import find_duplicates

all_data = {}
for spec in config.datasets:
    images = list_images(spec.root)
    stats = batch_compute_stats(images)
    dupes = find_duplicates(images, hamming_threshold=config.duplicate_hash_threshold).near_duplicates
    all_data[spec.name] = {"images": images, "stats": stats, "dupes": dupes}
    valid = [s for s in stats if not s.is_corrupted]
    print(f"\n{spec.name} — {len(images)} images, {len(valid)} valid, {sum(1 for s in stats if s.is_corrupted)} corrupted")
    print(f"  Widths:  {min(s.width for s in valid)}-{max(s.width for s in valid)} px")
    print(f"  Heights: {min(s.height for s in valid)}-{max(s.height for s in valid)} px")
    print(f"  Brightness: {min(s.brightness_mean for s in valid):.1f} / {sum(s.brightness_mean for s in valid)/len(valid):.1f} / {max(s.brightness_mean for s in valid):.1f}")
    print(f"  Blur (VoF): {min(s.blur_score for s in valid):.1f} / {sum(s.blur_score for s in valid)/len(valid):.1f} / {max(s.blur_score for s in valid):.1f}")
    print(f"  Entropy:    {min(s.entropy for s in valid):.2f} / {sum(s.entropy for s in valid)/len(valid):.2f} / {max(s.entropy for s in valid):.2f}")
    print(f"  Near-duplicate pairs: {len(dupes)}")


dataset_A — 464 images, 464 valid, 0 corrupted
  Widths:  256-3456 px
  Heights: 226-4608 px
  Brightness: 60.4 / 112.1 / 247.2
  Blur (VoF): 8.8 / 3169.2 / 20698.8
  Entropy:    1.68 / 7.49 / 7.81
  Near-duplicate pairs: 231


[07/08/26 10:37:16] WARNING  Could not compute phash for                                                           
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\data\raw\dataset_B\Vehicles\2018.jpg:
                             image file is truncated (5 bytes not processed)


dataset_B — 2087 images, 2086 valid, 1 corrupted
  Widths:  240-4032 px
  Heights: 249-3024 px
  Brightness: 13.6 / 101.1 / 180.4
  Blur (VoF): 3.6 / 663.9 / 12823.0
  Entropy:    3.79 / 7.26 / 7.96
  Near-duplicate pairs: 152


---
## 5. Part 2 — Generate EDA Figures

All figures are saved as **PNG + SVG** pairs into `reports/figures/<dataset_name>/`.
Inline previews are shown for key figures below.

In [8]:
from alpr_dataset.eda.figures import (
    plot_dataset_size_comparison,
    plot_resolution_histograms,
    plot_width_distribution,
    plot_height_distribution,
    plot_aspect_ratio_distribution,
    plot_brightness_histogram,
    plot_contrast_histogram,
    plot_blur_estimation,
    plot_sharpness,
    plot_entropy,
    plot_bbox_width_distribution,
    plot_bbox_height_distribution,
    plot_bbox_area_distribution,
    plot_bbox_position_heatmap,
    plot_class_distribution,
    plot_example_images,
    plot_random_samples,
    plot_annotated_samples,
    plot_color_distribution,
    plot_duplicate_visualization,
    plot_outlier_visualization,
    generate_all_eda_figures,
)
from alpr_dataset.eda.report import generate_eda_report

shared_fig_dir = config.reports_dir / "figures"

In [9]:
# Cross-dataset comparison
dataset_counts = {name: len(d["images"]) for name, d in all_data.items()}
plot_dataset_size_comparison(dataset_counts, shared_fig_dir)
print(f"Cross-dataset comparison saved to {shared_fig_dir}")

[07/08/26 10:37:20] INFO     Saved figure: dataset_size_comparison (.png/.svg)

Cross-dataset comparison saved to C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\figures


In [10]:
# Generate all figures for each dataset
for spec in config.datasets:
    d = all_data[spec.name]
    output_dir = shared_fig_dir / spec.name
    
    generate_all_eda_figures(
        dataset_counts={spec.name: len(d["images"])},
        stats=d["stats"],
        annotations=all_annotations[spec.name],
        image_paths=d["images"],
        duplicate_pairs=d["dupes"],
        output_dir=output_dir,
        blur_threshold=config.blur_threshold,
    )
    
    generate_eda_report(
        dataset_name=spec.name,
        stats=d["stats"],
        annotations=all_annotations[spec.name],
        image_paths=d["images"],
        duplicate_pairs=d["dupes"],
        figures_dir=output_dir,
        output_dir=shared_fig_dir,
    )
    print(f"{spec.name}: {len(d['images'])} images, figures in {output_dir}")

                    INFO     Saved figure: dataset_size_comparison (.png/.svg)

                    INFO     Saved figure: resolution_histograms (.png/.svg)

                    INFO     Saved figure: width_distribution (.png/.svg)

                    INFO     Saved figure: height_distribution (.png/.svg)

[07/08/26 10:37:21] INFO     Saved figure: aspect_ratio_distribution (.png/.svg)

...

[07/08/26 10:39:26] INFO     Saved figure: color_distribution (.png/.svg)

[07/08/26 10:39:30] INFO     Saved figure: duplicate_visualization (.png/.svg)

                    INFO     Saved figure: outlier_visualization (.png/.svg)

                    INFO     EDA report written:                                                                   
                             C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\figures\dataset_B_eda_report.
                             md

dataset_B: 2087 images, figures in C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\figures\dataset_B


---
## 6. Inline Figure Previews

Key figures displayed inline for quick visual inspection.

### 6.1 Resolution Distribution

The distribution of image widths and heights across the dataset.

In [11]:
# Pick the first dataset for inline preview
spec = config.datasets[0]
d = all_data[spec.name]
valid = [s for s in d["stats"] if not s.is_corrupted]
widths = [s.width for s in valid]
heights = [s.height for s in valid]
ratios = [w/h for w,h in zip(widths, heights) if h>0]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths, bins=40, color="#3d5a80", edgecolor="white")
axes[0].set_xlabel("Width (px)"); axes[0].set_ylabel("Frequency"); axes[0].set_title(f"{spec.name}: Width Distribution")
axes[1].hist(heights, bins=40, color="#ee6c4d", edgecolor="white")
axes[1].set_xlabel("Height (px)"); axes[1].set_ylabel("Frequency"); axes[1].set_title(f"{spec.name}: Height Distribution")
fig.tight_layout(); plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_36912\3463153103.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


### 6.2 Aspect Ratio Distribution

In [12]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(ratios, bins=40, color="#98c1d9", edgecolor="white")
ax.set_xlabel("Aspect ratio (width/height)")
ax.set_ylabel("Frequency")
ax.set_title(f"{spec.name}: Aspect Ratio Distribution")
fig.tight_layout(); plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_36912\2664031698.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


### 6.3 Photometric Quality

Brightness, contrast, blur estimation, sharpness, and entropy histograms.

In [13]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

brightness = [s.brightness_mean for s in valid]
contrast = [s.contrast_std for s in valid]
blur = [s.blur_score for s in valid]
sharp = [s.sharpness_score for s in valid]
entropy = [s.entropy for s in valid]

axes[0,0].hist(brightness, bins=40, color="#f4d35e", edgecolor="white")
axes[0,0].set_title("Brightness"); axes[0,0].set_xlabel("Mean intensity")
axes[0,1].hist(contrast, bins=40, color="#ee6c4d", edgecolor="white")
axes[0,1].set_title("Contrast"); axes[0,1].set_xlabel("Std dev")
axes[0,2].hist(blur, bins=40, color="#3d5a80", edgecolor="white")
axes[0,2].axvline(100, color="red", linestyle="--", alpha=0.7)
axes[0,2].set_title("Blur (VoF)"); axes[0,2].set_xlabel("Variance of Laplacian")
axes[1,0].hist(sharp, bins=40, color="#98c1d9", edgecolor="white")
axes[1,0].set_title("Sharpness"); axes[1,0].set_xlabel("Sobel magnitude")
axes[1,1].hist(entropy, bins=40, color="#293241", edgecolor="white")
axes[1,1].set_title("Entropy"); axes[1,1].set_xlabel("Shannon entropy (bits)")
axes[1,2].axis("off")
fig.suptitle(f"{spec.name}: Photometric Quality", fontsize=14)
fig.tight_layout(); plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_36912\2591291923.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


### 6.4 Bounding Box Analysis

Width, height, area distributions and position heatmap.

In [14]:
annots = all_annotations[spec.name]
bws = [b.width for a in annots for b in a.boxes if b.width>0]
bhs = [b.height for a in annots for b in a.boxes if b.height>0]
bas = [b.area for a in annots for b in a.boxes if b.area>0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(bws, bins=40, color="#3d5a80", edgecolor="white")
axes[0].set_title("BBox Width"); axes[0].set_xlabel("Width (px)")
axes[1].hist(bhs, bins=40, color="#ee6c4d", edgecolor="white")
axes[1].set_title("BBox Height"); axes[1].set_xlabel("Height (px)")
axes[2].hist(bas, bins=40, color="#4ba36f", edgecolor="white")
axes[2].set_title("BBox Area"); axes[2].set_xlabel("Area (px²)")
fig.suptitle(f"{spec.name}: Bounding Box Size Distributions", fontsize=13)
fig.tight_layout(); plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_36912\2659639138.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


In [15]:
# BBox Position Heatmap
import numpy as np
heat = np.zeros((32, 32), dtype=np.float64)
for ann in annots:
    if ann.image_width <= 0 or ann.image_height <= 0:
        continue
    for box in ann.boxes:
        cx, cy = box.center
        gx = min(int(cx / ann.image_width * 32), 31)
        gy = min(int(cy / ann.image_height * 32), 31)
        heat[gy, gx] += 1

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(heat, cmap="inferno", origin="upper")
ax.set_title(f"{spec.name}: BBox Position Heatmap")
plt.colorbar(im, ax=ax, label="Box centre count")
fig.tight_layout(); plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_36912\915443206.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


### 6.5 Sample Images

In [16]:
import random
import cv2
from alpr_dataset.io_utils import safe_read_image
from alpr_dataset.utils.viz_utils import bgr_to_rgb, draw_boxes

# Show random annotated samples
rng = random.Random(42)
with_boxes = [a for a in annots if a.boxes]
sample = rng.sample(with_boxes, min(6, len(with_boxes)))

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for idx, (ann, ax) in enumerate(zip(sample, axes.flat)):
    img = safe_read_image(ann.image_path)
    if img is None:
        continue
    img_rgb = bgr_to_rgb(img)
    for box in ann.boxes:
        import matplotlib.patches as patches
        rect = patches.Rectangle(
            (box.x_min, box.y_min), box.width, box.height,
            fill=False, edgecolor="#ee6c4d", linewidth=2
        )
        ax.add_patch(rect)
    ax.imshow(img_rgb)
    ax.set_title(ann.image_path.name, fontsize=8)
    ax.axis("off")
fig.suptitle(f"{spec.name}: Annotated Samples", fontsize=14)
fig.tight_layout(); plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_36912\291773712.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


### 6.6 Outlier Detection

Scatter plot of resolution vs file size with z-score outliers highlighted.

In [17]:
resolutions = np.array([s.width * s.height for s in valid])
sizes = np.array([s.file_size_bytes for s in valid])

z_res = (resolutions - resolutions.mean()) / (resolutions.std() or 1)
z_size = (sizes - sizes.mean()) / (sizes.std() or 1)
outlier = (np.abs(z_res) > 3) | (np.abs(z_size) > 3)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(resolutions[~outlier], sizes[~outlier], s=15, alpha=0.5, color="#3d5a80", label="Normal")
ax.scatter(resolutions[outlier], sizes[outlier], s=30, color="#ee6c4d", edgecolor="black", linewidth=0.5, label=f"Outlier ({outlier.sum()})")
ax.set_xlabel("Resolution (W×H, px)"); ax.set_ylabel("File size (bytes)")
ax.set_title(f"{spec.name}: Outlier Detection"); ax.legend()
fig.tight_layout(); plt.show()

C:\Users\Admin\AppData\Local\Temp\ipykernel_36912\2096004515.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


---
## 7. Validate Outputs

Check that every expected figure file exists as both PNG and SVG.

In [18]:
EXPECTED_FIGURES = [
    "resolution_histograms", "width_distribution", "height_distribution",
    "aspect_ratio_distribution", "brightness_histogram", "contrast_histogram",
    "blur_estimation", "sharpness_distribution", "entropy_distribution",
    "bbox_size_distribution", "bbox_width_distribution", "bbox_height_distribution",
    "bbox_area_distribution", "bbox_position_heatmap", "class_distribution",
    "example_images", "random_samples", "annotated_samples", "color_distribution",
    "duplicate_visualization", "outlier_visualization",
]

missing = []
for spec in config.datasets:
    fig_dir = shared_fig_dir / spec.name
    for stem in EXPECTED_FIGURES:
        if not (fig_dir / f"{stem}.png").is_file():
            missing.append(f"{spec.name}/{stem}.png")
        if not (fig_dir / f"{stem}.svg").is_file():
            missing.append(f"{spec.name}/{stem}.svg")

cd_fig = shared_fig_dir / "dataset_size_comparison"
if not (shared_fig_dir / "dataset_size_comparison.png").is_file():
    missing.append("dataset_size_comparison.png")

if missing:
    print(f"MISSING ({len(missing)}):")
    for m in missing:
        print(f"  - {m}")
else:
    total = len(config.datasets) * len(EXPECTED_FIGURES) * 2 + 2
    print(f"✅ All {total} figure files validated (PNG + SVG)")

✅ All 86 figure files validated (PNG + SVG)


---
## 8. Summary

| Dataset | Images | Valid | Corrupted | BBoxes | Near-Dupes |
|---------|--------|-------|-----------|--------|------------|
{% for spec in config.datasets -%}
{% set d = all_data[spec.name] -%}
{% set a = all_annotations[spec.name] -%}
| {{ spec.name }} | {{ d['images']|length }} | {{ d['stats']|selectattr('is_corrupted','equalto',False)|list|length }} | {{ d['stats']|selectattr('is_corrupted')|list|length }} | {{ a|sum(attribute='n_boxes') }} | {{ d['dupes']|length }} |
{% endfor %}

All figures saved to `reports/figures/`. EDA reports saved to `reports/figures/*_eda_report.md`.

In [19]:
# Print summary table
print(f"{'Dataset':<12} {'Images':>8} {'Valid':>8} {'Corrupt':>8} {'BBoxes':>8} {'NearDupes':>10}")
print("-" * 56)
for spec in config.datasets:
    d = all_data[spec.name]
    a = all_annotations[spec.name]
    n_valid = sum(1 for s in d["stats"] if not s.is_corrupted)
    n_corrupt = sum(1 for s in d["stats"] if s.is_corrupted)
    n_boxes = sum(aa.n_boxes for aa in a)
    n_dupes = len(d["dupes"])
    print(f"{spec.name:<12} {len(d['images']):>8} {n_valid:>8} {n_corrupt:>8} {n_boxes:>8} {n_dupes:>10}")

print(f"\n✅ EDA complete. Figures: {shared_fig_dir}")

Dataset        Images    Valid  Corrupt   BBoxes  NearDupes
--------------------------------------------------------
dataset_A         464      464        0      272        231
dataset_B        2087     2086        1     2142        152

✅ EDA complete. Figures: C:\Users\Admin\Documents\GitHub\AI-Tools-Project\reports\figures
